# Lag vs. Current Driver Model: Food Insecurity Comparison

This notebook compares three specifications of the ZIP/ZCTA-county food insecurity
transition model built in `1.3-lk-zipcode-food-insecurity-forecast-enhanced-variables.ipynb`:

1. **Current model** &mdash; `food_insecurity_rate[t] ~ lag_food_insecurity_rate[t-1] + drivers[t]`
   (the original repo specification: contemporaneous economic/demographic drivers plus last
   year's rate).
2. **2-year lag model** &mdash; `food_insecurity_rate[t] ~ food_insecurity_rate[t-2] + drivers[t-2]`
3. **3-year lag model** &mdash; `food_insecurity_rate[t] ~ food_insecurity_rate[t-3] + drivers[t-3]`

Both lag models use predictors and the lagged rate from the *same* past year, matching a
real forecasting constraint: if the most recent complete data you have is from `t-k`, you
don't have a fresher rate to lean on either, only that same lagged snapshot.

**Data caveat:** the observed MMG ZCTA panel only covers a handful of years. A 2-year lag
uses at most two ZIP-year transition pairs per row; a 3-year lag may use only one. Where
there isn't enough time-series depth to hold out a full year for validation, this notebook
automatically falls back to 5-fold cross-validation across ZIP-county rows and says so
explicitly in the output. Treat the 3-year lag results as more fragile than the others until
more MMG years are released.

All data loading and feature engineering below is copied from notebook 1.3 so the three
models are trained on an identical panel and predictor set &mdash; the only thing that changes
between specifications is *when* the predictors and lagged rate are measured relative to the
target year.


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

from openpyxl import load_workbook


## Configuration

In [2]:
# Resolve project paths so the notebook works from the repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "external").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EXTERNAL_DIR = PROJECT_ROOT / "data" / "external"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Main source workbook and sheets, same as notebook 1.3.
MMG_WORKBOOK_PATH = EXTERNAL_DIR / "MMG_2025.xlsx"
ZCTA_SHEET = "ZCTA - Zip code level data "
COUNTY_SHEET = "County"
ALICE_COUNTY_PATH = EXTERNAL_DIR / "2025 ALICE - Florida Data Sheet (Lee).xlsx - County.csv"

# Default: evaluate Feeding Tampa Bay rows as the local slice. Set to None to use all rows.
TARGET_FOOD_BANK = "Feeding Tampa Bay"

OUTPUT_PATH = PROCESSED_DIR / "lag_vs_current_model_comparison.csv"

MMG_WORKBOOK_PATH, ALICE_COUNTY_PATH, OUTPUT_PATH


(WindowsPath('C:/Users/Tom/ftb-pro-bono/data/external/MMG_2025.xlsx'),
 WindowsPath('C:/Users/Tom/ftb-pro-bono/data/external/2025 ALICE - Florida Data Sheet (Lee).xlsx - County.csv'),
 WindowsPath('C:/Users/Tom/ftb-pro-bono/data/processed/lag_vs_current_model_comparison.csv'))

## Helper Functions

In [3]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    """Normalize spreadsheet column names for consistent downstream access."""
    out = df.copy()
    out.columns = [re.sub(r"\s+", " ", str(c).strip()) if c is not None else f"unnamed_{i}" for i, c in enumerate(out.columns)]
    return out


def read_excel_sheet_openpyxl(path: Path, sheet_name: str) -> pd.DataFrame:
    """Read a workbook sheet directly with openpyxl to avoid pandas/openpyxl version issues."""
    wb = load_workbook(path, read_only=True, data_only=True)
    ws = wb[sheet_name]
    rows = ws.iter_rows(values_only=True)
    header = next(rows)

    # Drop trailing blank workbook columns so they do not become unusable dataframe fields.
    useful_cols = [i for i, value in enumerate(header) if value is not None]
    names = [header[i] for i in useful_cols]
    # Some sheets have a long tail of fully-blank rows past the real data (formatting artifacts);
    # skip any row that has no populated cells at all rather than assuming it matches the header width.
    data = [[row[i] for i in useful_cols] for row in rows if any(v is not None for v in row)]
    return clean_columns(pd.DataFrame(data, columns=names))


def parse_number(series: pd.Series) -> pd.Series:
    """Convert currency, comma-formatted, and percent-looking strings to numeric values."""
    cleaned = (
        series.astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.replace("%", "", regex=False)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
    )
    return pd.to_numeric(cleaned, errors="coerce")


def parse_rate(series: pd.Series) -> pd.Series:
    """Convert rate columns to decimals, e.g. 15.2% or 15.2 becomes 0.152."""
    values = parse_number(series)
    if values.dropna().max() > 1.5:
        values = values / 100.0
    return values


def clip_rate(series: pd.Series, low: float = 0.0, high: float = 0.95) -> pd.Series:
    """Keep modeled rates inside a plausible bounded interval."""
    return series.clip(lower=low, upper=high)


def fit_weighted_ridge(X: pd.DataFrame, y: pd.Series, weights: pd.Series, alpha: float = 1.0) -> dict:
    """Fit a population-weighted ridge regression using only NumPy."""
    # Standardize features so the ridge penalty treats each predictor comparably.
    means = X.mean()
    stds = X.std(ddof=0).replace(0, 1)
    X_scaled = (X - means) / stds

    # Add an intercept column and apply square-root weights for weighted least squares.
    X_design = np.column_stack([np.ones(len(X_scaled)), X_scaled.to_numpy(dtype=float)])
    y_values = y.to_numpy(dtype=float)
    w = np.sqrt(np.maximum(weights.fillna(weights.median()).to_numpy(dtype=float), 1.0))
    Xw = X_design * w[:, None]
    yw = y_values * w

    # Penalize feature coefficients but not the intercept.
    penalty = np.eye(X_design.shape[1]) * alpha
    penalty[0, 0] = 0
    coef = np.linalg.solve(Xw.T @ Xw + penalty, Xw.T @ yw)
    return {"features": list(X.columns), "means": means, "stds": stds, "coef": coef}


def predict_weighted_ridge(model: dict, X: pd.DataFrame) -> pd.Series:
    """Generate predictions from the fitted weighted ridge model."""
    X_scaled = (X[model["features"]] - model["means"]) / model["stds"]
    X_design = np.column_stack([np.ones(len(X_scaled)), X_scaled.to_numpy(dtype=float)])
    return pd.Series(X_design @ model["coef"], index=X.index)


def weighted_metrics(frame: pd.DataFrame, actual_col: str, pred_col: str, weight_col: str = "population") -> dict:
    """Population-weighted MAE / RMSE / mean error, matching the notebook 1.3 validation approach."""
    if frame.empty:
        return {"weighted_mae": np.nan, "weighted_rmse": np.nan, "weighted_mean_error": np.nan, "n": 0}
    w = frame[weight_col].fillna(1).clip(lower=1)
    err = frame[pred_col] - frame[actual_col]
    return {
        "weighted_mae": float(np.average(err.abs(), weights=w)),
        "weighted_rmse": float(np.sqrt(np.average(err ** 2, weights=w))),
        "weighted_mean_error": float(np.average(err, weights=w)),
        "n": int(len(frame)),
    }


## Load Source Data

In [4]:
# Load the national ZCTA panel, county context, and Florida ALICE hardship data.
zcta_raw = read_excel_sheet_openpyxl(MMG_WORKBOOK_PATH, ZCTA_SHEET)
county_raw = read_excel_sheet_openpyxl(MMG_WORKBOOK_PATH, COUNTY_SHEET)
alice_county_raw = clean_columns(pd.read_csv(ALICE_COUNTY_PATH, dtype=str))

len(zcta_raw), len(county_raw), len(alice_county_raw)


(1027, 50, 603)

## Build the ZIP-County-Year Panel

Identical to notebook 1.3: `row_id = state_fips + county_fips + zcta`, county-level Map the
Meal Gap context and Florida ALICE hardship variables joined on `county_fips` + `year`,
missing drivers filled with state medians then national medians.


In [5]:
# Work on copies so the raw imports remain untouched.
zcta = zcta_raw.copy()
county = county_raw.copy()
alice_county = alice_county_raw.copy()

# -----------------------------
# County-level feature table
# -----------------------------
county["county_fips"] = county["FIPS"].astype(str).str.zfill(5)
# The County sheet stores one row per county-year but does not include an explicit Year column.
# Rows are ordered 2019-2023 within each county, matching the Map the Meal Gap panel years.
county["_county_year_index"] = county.groupby("county_fips").cumcount()
county["year"] = county["_county_year_index"].map({0: 2019, 1: 2020, 2: 2021, 3: 2022, 4: 2023}).astype("Int64")
county["state_fips"] = county["county_fips"].str[:2]
county["county_population"] = parse_number(county["Total Population (5 Year ACS)"])
county["county_child_population"] = parse_number(county["Total Child Population (5 Year ACS)"])
county["county_child_population_share"] = county["county_child_population"] / county["county_population"]
county["county_percent_white_non_hispanic"] = parse_rate(county["Percent White, non-Hispanic (5 Year ACS)"])
county["county_cost_per_meal"] = parse_number(county["Cost Per Meal"])
county["county_weighted_food_cost_index"] = parse_number(county["Weighted Index"])
county["county_snap_threshold"] = parse_number(county["SNAP Threshold"])
county["county_rural_urban_code_2023"] = parse_number(county["Rural-Urban Continuum Code (2023)"])

county_features = county[[
    "county_fips", "year", "county_child_population_share", "county_percent_white_non_hispanic",
    "county_cost_per_meal", "county_weighted_food_cost_index", "county_snap_threshold",
    "county_rural_urban_code_2023"
]].dropna(subset=["county_fips", "year"]).copy()
county_features["year"] = county_features["year"].astype(int)

# -----------------------------
# Florida ALICE county features
# -----------------------------
alice_county["year"] = pd.to_numeric(alice_county["Year"], errors="coerce").astype("Int64")
alice_county["county_fips"] = alice_county["GEO id2"].astype(str).str.zfill(5)
alice_county["alice_households"] = parse_number(alice_county["ALICE Households"])
alice_county["alice_poverty_households"] = parse_number(alice_county["Poverty Households"])
alice_county["alice_total_households"] = parse_number(alice_county["Households"])
alice_county["alice_financial_insecurity_rate"] = (
    alice_county["alice_households"] + alice_county["alice_poverty_households"]
) / alice_county["alice_total_households"]
alice_county["alice_poverty_household_rate"] = alice_county["alice_poverty_households"] / alice_county["alice_total_households"]
alice_county["alice_threshold_under_65"] = parse_number(alice_county["ALICE Threshold - HH under 65"])
alice_county["alice_threshold_65_plus"] = parse_number(alice_county["ALICE Threshold - HH 65 years and over"])

alice_features = alice_county[[
    "county_fips", "year", "alice_financial_insecurity_rate", "alice_poverty_household_rate",
    "alice_threshold_under_65", "alice_threshold_65_plus"
]].dropna(subset=["county_fips", "year"]).copy()
alice_features["year"] = alice_features["year"].astype(int)

# -----------------------------
# ZCTA panel table
# -----------------------------
zcta["year"] = pd.to_numeric(zcta["Year"], errors="coerce").astype("Int64")
zcta["state_fips"] = zcta["State FIPS"].astype(str).str.zfill(2)
zcta["county_fips"] = zcta["County FIPS"].astype(str).str.zfill(5)
zcta["zcta"] = zcta["ZCTA"].astype(str).str.zfill(5)
zcta["row_id"] = zcta["state_fips"] + "_" + zcta["county_fips"] + "_" + zcta["zcta"]

zcta["population"] = parse_number(zcta["Total Population (5 Year ACS)"])
zcta["food_insecurity_rate"] = parse_rate(zcta["Overall Food Insecurity Rate"])
zcta["food_insecure_persons"] = parse_number(zcta["# of Food Insecure Persons Overall"])
zcta["unemployment_rate"] = parse_rate(zcta["Unemployment Rate (1 Yr BLS)"])
zcta["poverty_rate"] = parse_rate(zcta["Poverty Rate (5 Yr ACS)"])
zcta["percent_black"] = parse_rate(zcta["Percent Black (5 Yr ACS)"])
zcta["percent_hispanic"] = parse_rate(zcta["Percent Hispanic (any race) (5 Year ACS)"])
zcta["median_income"] = parse_number(zcta["Median Income (5 Yr ACS)"])
zcta["log_median_income"] = np.log(zcta["median_income"].replace(0, np.nan))
zcta["homeownership_rate"] = parse_rate(zcta["Homeownership Rate (5 Yr ACS)"])
zcta["disability_rate"] = parse_rate(zcta["Disability Rate (5 Yr ACS)"])

model_cols = [
    "row_id", "state_fips", "county_fips", "zcta", "Geography", "County, State", "State",
    "Food Bank 1 ID", "Food Bank 1", "Food Bank 2 ID", "Food Bank 2", "year", "population",
    "food_insecurity_rate", "food_insecure_persons", "unemployment_rate", "poverty_rate",
    "percent_black", "percent_hispanic", "median_income", "log_median_income",
    "homeownership_rate", "disability_rate"
]
panel = zcta[model_cols].dropna(subset=["year", "row_id"]).copy()
panel["year"] = panel["year"].astype(int)

panel = panel.merge(county_features, on=["county_fips", "year"], how="left")
panel = panel.merge(alice_features, on=["county_fips", "year"], how="left")
panel["has_alice_data"] = panel["alice_financial_insecurity_rate"].notna().astype(int)
panel = panel.sort_values(["row_id", "year"]).reset_index(drop=True)

# Fill driver gaps with state medians first, then national medians as a final fallback.
numeric_features = [
    "unemployment_rate", "poverty_rate", "percent_black", "percent_hispanic", "log_median_income",
    "homeownership_rate", "disability_rate", "county_child_population_share",
    "county_percent_white_non_hispanic", "county_cost_per_meal", "county_weighted_food_cost_index",
    "county_snap_threshold", "county_rural_urban_code_2023", "alice_financial_insecurity_rate",
    "alice_poverty_household_rate", "alice_threshold_under_65", "alice_threshold_65_plus", "has_alice_data"
]
for col in numeric_features:
    panel[col] = panel.groupby("State")[col].transform(lambda s: s.fillna(s.median()))
    panel[col] = panel[col].fillna(panel[col].median())

observed_years = sorted(panel["year"].dropna().unique().tolist())
print(f"Observed years: {observed_years}")
print(f"Rows: {len(panel):,}, unique row_ids: {panel['row_id'].nunique():,}")
display(panel.head())


Observed years: [2020, 2021, 2022, 2023]
Rows: 1,027, unique row_ids: 259


,row_id,state_fips,county_fips,zcta,Geography,"County, State",State,Food Bank 1 ID,Food Bank 1,Food Bank 2 ID,...,county_percent_white_non_hispanic,county_cost_per_meal,county_weighted_food_cost_index,county_snap_threshold,county_rural_urban_code_2023,alice_financial_insecurity_rate,alice_poverty_household_rate,alice_threshold_under_65,alice_threshold_65_plus,has_alice_data
0,12_12017_34428,12,12017,34428,ZCTA5 34428,"Citrus County, Florida",FL,90.0,Feeding Tampa Bay,None,...,0.876,3.63,1.12,2.0,1.0,0.469325,0.130295,64593.0,58284.0,0
1,12_12017_34428,12,12017,34428,ZCTA5 34428,"Citrus County, Florida",FL,90.0,Feeding Tampa Bay,None,...,0.869,3.87,1.08,2.0,1.0,0.531398,0.148105,60000.0,45000.0,1
2,12_12017_34428,12,12017,34428,ZCTA5 34428,"Citrus County, Florida",FL,90.0,Feeding Tampa Bay,None,...,0.862,4.24,1.06,2.0,3.0,0.525814,0.165513,60910.0,50088.0,1
3,12_12017_34428,12,12017,34428,ZCTA5 34428,"Citrus County, Florida",FL,90.0,Feeding Tampa Bay,None,...,0.857,3.76,1.05,2.0,3.0,0.505010,0.183299,56130.0,52296.0,1
4,12_12017_34429,12,12017,34429,ZCTA5 34429,"Citrus County, Florida",FL,90.0,Feeding Tampa Bay,None,...,0.876,3.63,1.12,2.0,1.0,0.469325,0.130295,64593.0,58284.0,0


## Driver Columns and Lag Specifications

In [6]:
# Same 18 drivers used in notebook 1.3, minus the lagged rate itself (added separately per spec).
driver_cols = [
    "unemployment_rate", "poverty_rate", "percent_black", "percent_hispanic",
    "log_median_income", "homeownership_rate", "disability_rate",
    "county_child_population_share", "county_percent_white_non_hispanic", "county_cost_per_meal",
    "county_weighted_food_cost_index", "county_snap_threshold", "county_rural_urban_code_2023",
    "alice_financial_insecurity_rate", "alice_poverty_household_rate", "alice_threshold_under_65",
    "alice_threshold_65_plus", "has_alice_data",
]

# (driver_lag, rate_lag): how many years back the drivers / lagged rate are measured from the target year.
# "current" reproduces notebook 1.3's specification: contemporaneous drivers, rate lagged one year.
LAG_SPECS = {
    "current (X[t], rate[t-1])": (0, 1),
    "2-year lag (X[t-2], rate[t-2])": (2, 2),
    "3-year lag (X[t-3], rate[t-3])": (3, 3),
}
LAG_SPECS


{'current (X[t], rate[t-1])': (0, 1),
 '2-year lag (X[t-2], rate[t-2])': (2, 2),
 '3-year lag (X[t-3], rate[t-3])': (3, 3)}

## Build a Lagged Transition Table

In [7]:
def build_transition_table(panel: pd.DataFrame, driver_lag: int, rate_lag: int, driver_cols: list) -> pd.DataFrame:
    """Pair each row's current-year target with drivers/rate measured `driver_lag`/`rate_lag` years earlier."""
    df = panel.sort_values(["row_id", "year"]).copy()
    g = df.groupby("row_id")

    out = df[["row_id", "year", "population", "food_insecurity_rate", "Food Bank 1", "State"]].copy()
    out["lag_food_insecurity_rate"] = g["food_insecurity_rate"].shift(rate_lag)
    for col in driver_cols:
        out[col] = g[col].shift(driver_lag)
    return out


## Fit, Tune, and Evaluate a Lag Specification

In [8]:
def evaluate_lag_spec(
    panel: pd.DataFrame,
    label: str,
    driver_lag: int,
    rate_lag: int,
    driver_cols: list,
    target_food_bank: str,
    alpha_grid=(0.01, 0.1, 0.5, 1.0, 2.0),
    n_folds: int = 5,
    random_state: int = 42,
) -> dict:
    """Fit a population-weighted ridge model for one lag specification and validate it.

    Uses a time-based holdout (train on earlier transition years, evaluate on the latest
    available one) whenever at least two transition years exist for this lag. Falls back to
    5-fold cross-validation across ZIP-county rows when the lag leaves only one usable
    transition year, since there is nothing earlier left to train on.
    """
    feature_cols = ["lag_food_insecurity_rate"] + driver_cols
    table = build_transition_table(panel, driver_lag, rate_lag, driver_cols)
    valid = table.dropna(subset=["food_insecurity_rate", "lag_food_insecurity_rate", "population"] + driver_cols).copy()

    if target_food_bank:
        valid["is_local"] = valid["Food Bank 1"].astype(str).str.contains(target_food_bank, case=False, na=False)
    else:
        valid["is_local"] = True

    available_years = sorted(valid["year"].unique().tolist())
    if len(available_years) == 0:
        raise ValueError(f"{label}: no usable transitions for driver_lag={driver_lag}, rate_lag={rate_lag}.")

    if len(available_years) >= 2:
        method = "time_holdout"
        latest_year = available_years[-1]
        train_full = valid.loc[valid["year"] < latest_year].copy()
        test = valid.loc[valid["year"] == latest_year].copy()

        alpha_rows = []
        for alpha in alpha_grid:
            m = fit_weighted_ridge(train_full[feature_cols], train_full["food_insecurity_rate"], train_full["population"], alpha=alpha)
            pred = clip_rate(predict_weighted_ridge(m, test[feature_cols]))
            scored = test.assign(prediction=pred)
            nat = weighted_metrics(scored, "food_insecurity_rate", "prediction")
            loc = weighted_metrics(scored.loc[scored["is_local"]], "food_insecurity_rate", "prediction")
            alpha_rows.append({"alpha": alpha, "national_weighted_mae": nat["weighted_mae"], "local_weighted_mae": loc["weighted_mae"]})
        alpha_df = pd.DataFrame(alpha_rows)
        selection_metric = "local_weighted_mae" if alpha_df["local_weighted_mae"].notna().any() else "national_weighted_mae"
        selected_alpha = float(alpha_df.sort_values([selection_metric, "national_weighted_mae", "alpha"]).iloc[0]["alpha"])

        final_model = fit_weighted_ridge(train_full[feature_cols], train_full["food_insecurity_rate"], train_full["population"], alpha=selected_alpha)
        pred = clip_rate(predict_weighted_ridge(final_model, test[feature_cols]))
        scored = test.assign(prediction=pred)
        national_metrics = weighted_metrics(scored, "food_insecurity_rate", "prediction")
        local_metrics = weighted_metrics(scored.loc[scored["is_local"]], "food_insecurity_rate", "prediction")
        train_years = sorted(train_full["year"].unique().tolist())
        n_train = len(train_full)
        n_test = len(test)

    else:
        method = "5_fold_cv_fallback"
        shuffled = valid.sample(frac=1.0, random_state=random_state).reset_index(drop=True)
        shuffled["fold"] = np.arange(len(shuffled)) % n_folds

        def run_cv(alpha):
            fold_frames = []
            for f in range(n_folds):
                train_fold = shuffled.loc[shuffled["fold"] != f]
                test_fold = shuffled.loc[shuffled["fold"] == f]
                m = fit_weighted_ridge(train_fold[feature_cols], train_fold["food_insecurity_rate"], train_fold["population"], alpha=alpha)
                pred = clip_rate(predict_weighted_ridge(m, test_fold[feature_cols]))
                fold_frames.append(test_fold.assign(prediction=pred))
            return pd.concat(fold_frames, ignore_index=True)

        alpha_rows = []
        for alpha in alpha_grid:
            cv_scored = run_cv(alpha)
            nat = weighted_metrics(cv_scored, "food_insecurity_rate", "prediction")
            loc = weighted_metrics(cv_scored.loc[cv_scored["is_local"]], "food_insecurity_rate", "prediction")
            alpha_rows.append({"alpha": alpha, "national_weighted_mae": nat["weighted_mae"], "local_weighted_mae": loc["weighted_mae"]})
        alpha_df = pd.DataFrame(alpha_rows)
        selection_metric = "local_weighted_mae" if alpha_df["local_weighted_mae"].notna().any() else "national_weighted_mae"
        selected_alpha = float(alpha_df.sort_values([selection_metric, "national_weighted_mae", "alpha"]).iloc[0]["alpha"])

        scored = run_cv(selected_alpha)
        national_metrics = weighted_metrics(scored, "food_insecurity_rate", "prediction")
        local_metrics = weighted_metrics(scored.loc[scored["is_local"]], "food_insecurity_rate", "prediction")

        # Final model fit on all available rows, for coefficient inspection only -- not what generated the metrics above.
        final_model = fit_weighted_ridge(shuffled[feature_cols], shuffled["food_insecurity_rate"], shuffled["population"], alpha=selected_alpha)
        train_years = available_years
        n_train = len(shuffled)
        n_test = len(scored)

    return {
        "label": label,
        "driver_lag": driver_lag,
        "rate_lag": rate_lag,
        "validation_method": method,
        "selected_alpha": selected_alpha,
        "available_years": available_years,
        "train_years": train_years,
        "n_train": n_train,
        "n_test": n_test,
        "national_weighted_mae": national_metrics["weighted_mae"],
        "national_weighted_rmse": national_metrics["weighted_rmse"],
        "national_weighted_mean_error": national_metrics["weighted_mean_error"],
        "national_n": national_metrics["n"],
        "local_weighted_mae": local_metrics["weighted_mae"],
        "local_weighted_rmse": local_metrics["weighted_rmse"],
        "local_weighted_mean_error": local_metrics["weighted_mean_error"],
        "local_n": local_metrics["n"],
        "model": final_model,
        "feature_cols": feature_cols,
    }


## Run All Three Specifications

If a specification prints `validation_method: 5_fold_cv_fallback`, there wasn't enough
time-series depth for a genuine out-of-time holdout at that lag -- treat its metrics as
lower-confidence than the ones using `time_holdout`.


In [9]:
results = {}
for label, (driver_lag, rate_lag) in LAG_SPECS.items():
    result = evaluate_lag_spec(
        panel=panel,
        label=label,
        driver_lag=driver_lag,
        rate_lag=rate_lag,
        driver_cols=driver_cols,
        target_food_bank=TARGET_FOOD_BANK,
    )
    results[label] = result
    print(
        f"{label}: method={result['validation_method']}, alpha={result['selected_alpha']:g}, "
        f"train_years={result['train_years']}, n_train={result['n_train']:,}, n_test={result['n_test']:,}"
    )


current (X[t], rate[t-1]): method=time_holdout, alpha=2, train_years=[2021, 2022], n_train=474, n_test=234
2-year lag (X[t-2], rate[t-2]): method=time_holdout, alpha=2, train_years=[2022], n_train=236, n_test=235


3-year lag (X[t-3], rate[t-3]): method=5_fold_cv_fallback, alpha=0.01, train_years=[2023], n_train=234, n_test=234


In [10]:
comparison_df = pd.DataFrame([
    {
        "specification": r["label"],
        "validation_method": r["validation_method"],
        "selected_alpha": r["selected_alpha"],
        "train_years": r["train_years"],
        "n_train": r["n_train"],
        "n_test": r["n_test"],
        "national_weighted_mae": r["national_weighted_mae"],
        "national_weighted_rmse": r["national_weighted_rmse"],
        "national_weighted_mean_error": r["national_weighted_mean_error"],
        "local_weighted_mae": r["local_weighted_mae"],
        "local_weighted_rmse": r["local_weighted_rmse"],
        "local_weighted_mean_error": r["local_weighted_mean_error"],
        "local_n": r["local_n"],
    }
    for r in results.values()
])
display(comparison_df)
comparison_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved comparison table to: {OUTPUT_PATH}")


,specification,validation_method,selected_alpha,train_years,n_train,n_test,national_weighted_mae,national_weighted_rmse,national_weighted_mean_error,local_weighted_mae,local_weighted_rmse,local_weighted_mean_error,local_n
0,"current (X[t], rate[t-1])",time_holdout,2.00,"[2021, 2022]",474,234,0.044678,0.044995,-0.044678,0.044678,0.044995,-0.044678,234
1,"2-year lag (X[t-2], rate[t-2])",time_holdout,2.00,[2022],236,235,0.047470,0.048640,-0.047426,0.047470,0.048640,-0.047426,235
2,"3-year lag (X[t-3], rate[t-3])",5_fold_cv_fallback,0.01,[2023],234,234,0.008679,0.011687,-0.000179,0.008679,0.011687,-0.000179,234


Saved comparison table to: C:\Users\Tom\ftb-pro-bono\data\processed\lag_vs_current_model_comparison.csv


## Compare Coefficients Across Specifications

Standardized ridge coefficients (each predictor scaled to mean 0 / std 1 before fitting), so
magnitudes are roughly comparable within a spec. Compare signs and relative sizes across
columns to see whether a driver's relationship with food insecurity holds up, flips, or fades
once it's measured 2-3 years before the target year instead of contemporaneously.


In [11]:
coef_frame = pd.DataFrame({"feature": ["intercept"] + results[list(LAG_SPECS)[0]]["feature_cols"]})
for label, r in results.items():
    coef_frame[label] = r["model"]["coef"]
display(coef_frame)


,feature,"current (X[t], rate[t-1])","2-year lag (X[t-2], rate[t-2])","3-year lag (X[t-3], rate[t-3])"
0,intercept,0.129169,0.143128,0.151015
1,lag_food_insecurity_rate,-0.000164,-0.032055,-0.013529
2,unemployment_rate,0.011640,0.018725,0.009865
3,poverty_rate,0.023477,0.033720,0.020367
4,percent_black,-0.008595,-0.012638,-0.005678
5,percent_hispanic,-0.002102,-0.005434,-0.001206
6,log_median_income,-0.000633,-0.010202,-0.010016
7,homeownership_rate,-0.009250,-0.016831,-0.014016
8,disability_rate,0.010159,0.012402,0.008165
9,county_child_population_share,-0.000744,-0.003605,-0.004302


## Notes and Limitations

- **This notebook compares model specifications; it does not (yet) produce a recursive
  2024+ forecast.** Once you've picked a specification, the recursive forecasting logic in
  notebook 1.3 (driver projection, 8-year recursive loop, uncertainty bands, output CSV) can
  be adapted to use lagged rather than contemporaneous drivers.
- The 2-year and especially 3-year lag specifications are trained on very few transition
  years given the current MMG panel length. A `5_fold_cv_fallback` result should be treated
  as a rough signal, not a robust validation -- re-run this notebook once additional years
  of Map the Meal Gap data are released and the lag specs pick up more `time_holdout` years.
- All three specifications share the exact same panel, driver definitions, and cleaning
  logic as notebook 1.3, so the comparison isolates the effect of *when* the predictors are
  measured, not differences in data prep.
- Lower `weighted_mae` / `weighted_rmse` and a `weighted_mean_error` close to 0 indicate a
  better-fitting, less-biased specification. Compare the `local_*` columns (Feeding Tampa Bay
  rows) since that's the service area this project targets, not just the `national_*` columns.
